# Day 3, Notebook 2: the rows a profile cannot see

Notebook 1 profiled the columns. Every count came back explainable except one: `order_id` holds 49
distinct values across 50 rows.

This notebook is about that missing one, about the order at the far end of the amount column, and
about what you ship at the end of the day.

Position in the day:

`[profile the columns] > [decide per field] > **[find the hidden rows]** > [investigate the extremes] > [ship with the log]`

## Setup

Same file, same functions, one cell so this runs cold in a fresh Codespace.

In [ ]:
import csv
import os

DATA_DIR = "../data"
ORDERS_CSV = f"{DATA_DIR}/C2_W01_D03_orders_STUDENT.csv"
COMPANION_CSV = f"{DATA_DIR}/C2_W01_D03_companion_STUDENT.csv"
OUTPUT_DIR = "output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(ORDERS_CSV) as f:
    orders = list(csv.DictReader(f))


def normalise_amount(raw):
    """Yesterday's function, unchanged."""
    return int(raw)


def clean_record(record):
    """Yesterday's function, unchanged."""
    keeper = dict(record)
    keeper["amount"] = normalise_amount(record["amount"])
    return keeper


def clean_records(rows):
    """Yesterday's function, unchanged."""
    clean, rejects = [], []
    for r in rows:
        try:
            clean.append(clean_record(r))
        except ValueError as e:
            rejects.append({"order_id": r["order_id"], "reason": str(e)})
    return clean, rejects

decisions = []

def record_decision(field, finding, choice, reason):
    decisions.append({"field": field, "finding": finding, "choice": choice, "reason": reason})

print(f"{len(orders)} orders loaded")

## Section 1: the duplicate check that finds nothing

The obvious way to look for duplicates is to ask whether any two rows are identical.

In [ ]:
whole_rows = [tuple(r.values()) for r in orders]
duplicates = len(whole_rows) - len(set(whole_rows))
print(f"duplicate rows by whole-record comparison: {duplicates}")

Zero. Clean dataset, move on.

Nothing raised. Nothing was highlighted. This is what a wrong answer looks like on a good day.

## Section 2: the count that disagrees

Section 1 plus one new element: a second number to compare the first against.

In [ ]:
ids = [r["order_id"] for r in orders]
print(f"rows: {len(ids)}")
print(f"distinct order_ids: {len(set(ids))}")
print(f"unexplained: {len(ids) - len(set(ids))}")

Two numbers, on two different lines, that disagree. Nothing in the code noticed, because nothing in
the code was asked to compare them.

That comparison is the whole of today's first lesson, and it costs one line.

In [ ]:
from collections import Counter

repeated = [oid for oid, n in Counter(ids).items() if n > 1]
print("order_ids appearing more than once:", repeated)
print()
for r in orders:
    if r["order_id"] in repeated:
        print({k: r[k] for k in ("order_id", "segment", "amount", "status", "order_date")})

Same order id. Same amount. Same status. Same segment. Six weeks apart.

So which is it?

The same order, exported twice, with the second export stamping the wrong date? Or a genuine repeat
order that reused an id it should not have?

The file cannot tell you and it never could.

## Section 3: an identity rule is something you state

Section 2 plus one new element: a rule, written down, that somebody else could apply.

"They look like duplicates" cannot be applied by anyone else. This can.

In [ ]:
def find_duplicates(rows, key_fields):
    """Group rows by a stated identity rule and return the groups holding more than one row."""
    groups = {}
    for r in rows:
        key = tuple(r[f] for f in key_fields)
        groups.setdefault(key, []).append(r)
    return {k: v for k, v in groups.items() if len(v) > 1}

for rule in (["order_id"],
             ["order_id", "order_date"],
             ["order_id", "amount", "status"],
             list(orders[0].keys())):
    found = find_duplicates(orders, rule)
    rows_removed = sum(len(v) - 1 for v in found.values())
    print(f"{' + '.join(rule):58} groups {len(found)}, rows it would remove {rows_removed}")

Four defensible rules, three different answers. The rule is the decision, and the number follows
from it rather than the other way round.

Note the last line: the whole-record rule finds nothing, which is where section 1 started.

### Who decides

Not you, on your own, on a Wednesday.

The pair goes to whoever owns the order book, with both rows on screen and your proposed rule
underneath. Until then the pair is flagged rather than deleted, and the flag is a decision too.

In [ ]:
record_decision("order_id",
                f"{len(orders) - len(set(ids))} order_id shared by two rows ({repeated[0]})",
                "keep both rows, flag the pair",
                "the rows differ on order_date by six weeks, so re-export and repeat order are both "
                "plausible, and the order book owner decides which")

print(decisions[-1])

### Milestone: where this shows up in production

Every warehouse you will work in has a documented grain, meaning the statement of what one row
represents. "One row per order per export batch" and "one row per order" are different grains and
they disagree by exactly the pair you just found. Arguments about double-counted revenue are almost
always arguments about grain.

### Interview question this milestone just made answerable

"Two records share an id and disagree in one field. What do you do, and who decides?"

State an identity rule, flag rather than delete, and take it to whoever owns the data. The answer
that gets you rejected is deleting one of them because it looked like a duplicate.

## Section 4: the order at the end of the column

Section 3 plus one new element: the extremes.

Sort the column and read the tail. That is the entire technique, and it needs no arithmetic.

In [ ]:
clean, rejects = clean_records(orders)
amounts = sorted(r["amount"] for r in clean)

print("smallest five:", amounts[:5])
print("largest five: ", amounts[-5:])

Four ordinary orders and then one that is over 160 times the one before it.

In [ ]:
total = sum(amounts)
whale = amounts[-1]
print(f"orders that convert: {len(amounts)}")
print(f"total:               Rs {total}")
print(f"largest order:       Rs {whale}")
print(f"its share of the total: {round(100 * whale / total)} percent")
print(f"total without it:    Rs {total - whale}")

One order out of fifty is 86 percent of the money in the file.

The instinct is to delete it, because it is ruining every number you might compute.

### Why the instinct is wrong

An outlier is a finding before it is a row. In a business where a typical order sits between Rs 800
and Rs 3,000, an order of Rs 480,000 is either the most interesting customer in this file or a data
entry error, and those two need opposite responses.

So look at it rather than at its size.

In [ ]:
for r in clean:
    if r["amount"] == whale:
        print(r)

It converts cleanly. It has a customer, a date, a segment and a status like every other order.
Nothing about it is malformed. It is simply large.

So it survives cleaning, and it goes to tomorrow as a question rather than as a deletion.

In [ ]:
record_decision("amount",
                f"one order at Rs {whale}, {round(100 * whale / total)} percent of the total",
                "keep, and raise with the order book owner",
                "it converts cleanly and is well formed, so it is real until somebody says otherwise")

# A simple fence, with no arithmetic beyond a multiple of the middle value.
middle = amounts[len(amounts) // 2]
fence = middle * 10
flagged = [a for a in amounts if a > fence]
print(f"middle order Rs {middle}, fence at ten times that is Rs {fence}")
print(f"orders above the fence: {flagged}")

The fence is a convenience for spotting the tail quickly. It is not a test and it proves nothing.
The sorted tail on its own would have found the same order.

### Interview question

"The whale survived cleaning. Why?"

Because cleaning removes what cannot be read, and that order reads perfectly. Removing it would be
an analysis decision wearing a cleaning decision's clothes.

## Section 5: what ships

Section 4 plus one new element: the two files that leave this notebook.

The profiled dataset without the decisions log is an opinion.

In [ ]:
FIELDS = list(orders[0].keys())

clean_path = f"{OUTPUT_DIR}/C2_W01_D03_profiled_orders_STUDENT.csv"
rejects_path = f"{OUTPUT_DIR}/C2_W01_D03_rejects_STUDENT.csv"
log_path = f"{OUTPUT_DIR}/C2_W01_D03_decisions_log_STUDENT.csv"

with open(clean_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=FIELDS); w.writeheader(); w.writerows(clean)

with open(rejects_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["order_id", "reason"]); w.writeheader(); w.writerows(rejects)

with open(log_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["field", "finding", "choice", "reason"])
    w.writeheader(); w.writerows(decisions)

for p in (clean_path, rejects_path, log_path):
    print("wrote", p)

### The reconciliation, one level up

Yesterday it was input equals clean plus rejected. Today it also has to survive every decision you
made, including the pair you chose to keep.

In [ ]:
with open(clean_path) as f:
    back_clean = list(csv.DictReader(f))
with open(rejects_path) as f:
    back_rejects = list(csv.DictReader(f))
with open(log_path) as f:
    back_log = list(csv.DictReader(f))

print(f"{len(orders)} in = {len(back_clean)} profiled + {len(back_rejects)} rejected")
assert len(back_clean) + len(back_rejects) == len(orders), "orders went missing"
print(f"decisions recorded: {len(back_log)}")
for d in back_log:
    print(f"  {d['field']:10} {d['choice']}")

Three files, read back from disk, and the counts add up. That is a defensible day's work.

## Section 6: the file that breaks the contract

One last thing arrives from the same upstream system: a companion export with the same fields.

In [ ]:
with open(COMPANION_CSV) as f:
    companion = list(csv.DictReader(f))

print(f"rows read: {len(companion)}")
print("first row:", companion[0])

Read the first row again.

`DictReader` took its keys from line 1 and then handed you line 2 as data, and line 2 is another
copy of the header. So your first "order" has an `order_id` of `order_id`.

Nothing raised, because a header row is a perfectly valid row of text.

In [ ]:
suspects = [r for r in companion if r["order_id"] == "order_id"]
print(f"rows that are actually a repeated header: {len(suspects)}")

usable = [r for r in companion if r["order_id"] != "order_id"]
print(f"rows after removing them: {len(usable)}")

record_decision("companion file",
                "line 2 repeats the header row",
                "drop rows whose order_id reads order_id, and tell the sender",
                "a repeated header is a export defect, not data, and it would become an order")
print(decisions[-1])

This is the same lesson as the profile: the check that would have caught it is a comparison nobody
thought to make. `distinct` on `order_id` would have shown it too.

## Crux

A whole-record check finds only exact copies, so compare the id count against the row count every
time.

An identity rule is something you state, and whoever owns the data decides it.

An outlier is a finding to investigate before it is a row to delete.

The profiled dataset without its decisions log is an opinion.

## What tomorrow does with this

You have a dataset you can defend and a whale you decided to keep. Tomorrow you compute the average
order value and find out what that whale does to it.

Keep your three output files and your functions.